In [5]:

import os, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import densenet121
from sklearn.model_selection import train_test_split


# Backend setup

torch.manual_seed(42)
np.random.seed(42)
torch.backends.quantized.engine = "fbgemm"
device = torch.device("cpu")   # PTQ runs/evaluates on CPU



# Dataset & DataLoaders

transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"   # <-- change this
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
print(f"Total images: {len(dataset)}, Classes: {len(dataset.classes)}")

# Stratified split
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for cid in np.unique(targets):
    idxs = np.where(targets == cid)[0]
    train_ids, temp_ids = train_test_split(idxs, test_size=0.15,
                                           random_state=42, shuffle=True)
    val_ids, test_ids = train_test_split(temp_ids, test_size=1/3,
                                         random_state=42, shuffle=True)
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

batch_size = 128
num_workers = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
print("DataLoaders ready")


#  Load trained FP32 DenseNet121

NUM_CLASSES = len(dataset.classes)  # 200
model_fp32 = densenet121(weights=None)
model_fp32.classifier = nn.Linear(model_fp32.classifier.in_features, NUM_CLASSES)

# Load trained baseline weights
state_dict = torch.load("densenet121_weights.pth", map_location="cpu")
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

missing, unexpected = model_fp32.load_state_dict(new_state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)
print("FP32 DenseNet121 loaded")

model_fp32.eval()



# AdaRound utilities

def adaround_weight_tensor(weight, n_bits=8, num_iters=500, lr=1e-2):
    """
    Perform AdaRound optimization for a single weight tensor.
    """
    w = weight.detach().clone()
    scale = w.abs().max() / (2 ** (n_bits - 1) - 1)
    alpha = torch.zeros_like(w, requires_grad=True)

    opt = torch.optim.Adam([alpha], lr=lr)
    for _ in range(num_iters):
        opt.zero_grad()
        # quantize with learned rounding
        w_q = torch.round(w / scale + torch.tanh(alpha)) - torch.tanh(alpha)
        soft_w = w_q * scale
        loss = torch.nn.functional.mse_loss(soft_w, w)
        loss.backward()
        opt.step()

    with torch.no_grad():
        w_q = torch.round(w / scale + torch.tanh(alpha)) - torch.tanh(alpha)
        return (w_q * scale).detach()


def apply_adaround(model, n_bits=8, num_iters=500, lr=1e-2):

    m = copy.deepcopy(model)
    for name, module in m.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and module.weight is not None:
            print(f"AdaRounding: {name}.weight")
            q_w = adaround_weight_tensor(module.weight, n_bits=n_bits,
                                         num_iters=num_iters, lr=lr)
            with torch.no_grad():
                module.weight.copy_(q_w)
    return m



# Evaluation

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total


def get_model_size(model, filename="temp.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb


def benchmark(model, loader, num_batches=50):
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            _ = model(images)
    end = time.time()
    total_imgs = num_batches * loader.batch_size
    ms_per_batch = (end - start) / num_batches * 1000.0
    ms_per_image = (end - start) / total_imgs * 1000.0
    return ms_per_batch, ms_per_image



# Run AdaRound

model_ada = apply_adaround(model_fp32, n_bits=8, num_iters=500, lr=1e-2)

acc_fp32 = evaluate(model_fp32, test_loader)
acc_ada  = evaluate(model_ada,  test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"AdaRound INT8 Accuracy: {acc_ada:.2f}%")



# Size & Runtime

fp32_size = get_model_size(model_fp32)
ada_size  = get_model_size(model_ada)
print(f"FP32 size: {fp32_size:.2f} MB | AdaRound size: {ada_size:.2f} MB")

b_ms_fp32, i_ms_fp32 = benchmark(model_fp32, test_loader)
b_ms_ada,  i_ms_ada  = benchmark(model_ada,  test_loader)
print(f"FP32 Inference: {b_ms_fp32:.2f} ms/batch | {i_ms_fp32:.2f} ms/image")
print(f"AdaRound Inference: {b_ms_ada:.2f} ms/batch | {i_ms_ada:.2f} ms/image")




Total images: 100000, Classes: 200
Train: 85000, Val: 10000, Test: 5000
✅ DataLoaders ready
Missing keys: []
Unexpected keys: []
✅ FP32 DenseNet121 loaded
AdaRounding: features.conv0.weight
AdaRounding: features.denseblock1.denselayer1.conv1.weight
AdaRounding: features.denseblock1.denselayer1.conv2.weight
AdaRounding: features.denseblock1.denselayer2.conv1.weight
AdaRounding: features.denseblock1.denselayer2.conv2.weight
AdaRounding: features.denseblock1.denselayer3.conv1.weight
AdaRounding: features.denseblock1.denselayer3.conv2.weight
AdaRounding: features.denseblock1.denselayer4.conv1.weight
AdaRounding: features.denseblock1.denselayer4.conv2.weight
AdaRounding: features.denseblock1.denselayer5.conv1.weight
AdaRounding: features.denseblock1.denselayer5.conv2.weight
AdaRounding: features.denseblock1.denselayer6.conv1.weight
AdaRounding: features.denseblock1.denselayer6.conv2.weight
AdaRounding: features.transition1.conv.weight
AdaRounding: features.denseblock2.denselayer1.conv1.weig